# Projeto: Seleção de Região para Desenvolvimento de Poços de Petróleo — OilyGiant

## Introdução

A **OilyGiant**, empresa de mineração e exploração de petróleo, está avaliando três regiões candidatas para o desenvolvimento de novos poços. O objetivo deste projeto é construir um processo analítico completo — baseado em aprendizado de máquina e simulação estatística — que permita recomendar, de forma justificada, a região com o maior potencial de lucro e risco financeiro aceitável.

### Dados disponíveis

Foram fornecidos três conjuntos de dados, um por região (`geo_data_0.csv`, `geo_data_1.csv` e `geo_data_2.csv`), cada um contendo aproximadamente **100.000 registros** de poços já explorados, com as seguintes colunas:

- **id**: identificador único do poço;
- **f0, f1, f2**: três características numéricas obtidas em estudos geológicos/sísmicos preliminares. Seu significado físico específico não é divulgado, mas cada uma possui relação estatística relevante com o volume de reservas do poço;
- **product**: volume de reservas do poço, em milhares de barris — variável que queremos estimar para poços ainda não perfurados.

### Regras de negócio

- Apenas **regressão linear** deve ser utilizada para o treinamento dos modelos preditivos;
- Em cada região, o processo de exploração avalia **500 pontos** por vez, dos quais os **200 melhores** (maior volume estimado) são selecionados para desenvolvimento;
- O orçamento disponível é de **100 milhões de dólares** para o desenvolvimento desses 200 poços;
- Cada unidade de `product` (mil barris) gera uma receita de **4.500 dólares**;
- Consequentemente, cada poço desenvolvido precisa produzir, em média, ao menos **≈111,1 mil barris** para que o investimento não resulte em prejuízo;
- Somente serão consideradas viáveis as regiões com **risco de prejuízo inferior a 2,5%**; entre as regiões aprovadas nesse critério, a escolhida será a de maior **lucro médio esperado**.

### Abordagem

1. **Modelagem preditiva** — para cada região, treinar um modelo de regressão linear que estime `product` a partir de `f0`, `f1` e `f2`, utilizando 75% dos dados para treino e 25% para validação. A qualidade de cada modelo será avaliada pelo **REQM** e pelo **volume médio previsto**.
2. **Simulação da seleção de poços** — usando as predições no conjunto de validação, simular a escolha dos 200 melhores poços, calculando o lucro potencial.
3. **Avaliação de risco via Bootstrapping** — com **1.000 reamostragens**, gerar a distribuição de lucros de cada região, calculando lucro médio, intervalo de confiança de 95% e risco de prejuízo.
4. **Recomendação final** — indicar a região mais adequada, respeitando o critério de risco inferior a 2,5%.

## 1. Abertura e Exploração dos Dados

In [1]:
# Importando as bibliotecas
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import pandas as pd
from sklearn.dummy import DummyRegressor
import numpy as np

In [2]:
# Abertura dos dados
geo_data_0 = pd.read_csv('/datasets/geo_data_0.csv')
geo_data_1 = pd.read_csv('/datasets/geo_data_1.csv')
geo_data_2 = pd.read_csv('/datasets/geo_data_2.csv')

### 1.1 Região 0

In [3]:
# Dados da Região 0

print('SAMPLE DE DADOS ALEATÓRIOS DA PLANILHA GEO DATA 0')
print(geo_data_0.sample(10))
print('________________________________________________')
print('INFORMAÇÕES DA PLANILHA')
print(geo_data_0.info())
print('________________________________________________')
print('DESCRIÇÃO DE DADOS NUMÉRICOS DA PLANILHA')
print(geo_data_0.describe())
print('________________________________________________')
# Verificando duplicatas
print('LINHAS DUPLICADAS:', geo_data_0.duplicated().sum())

SAMPLE DE DADOS ALEATÓRIOS DA PLANILHA GEO DATA 0
,          id        f0        f1        f2     product
,29047  1hAR0  0.396567  1.000448 -7.402968    7.676429
,63313  3lY3Y  0.909668  0.257622  3.666265  130.140838
,27530  OUYI3 -0.904442  0.213181  4.470671  107.103638
,88052  8TK5l  0.863484 -0.374970 -0.519677   56.652661
,87638  nkKRZ -0.923868  0.367148  1.020890   67.661703
,24159  cqRNT  2.018568  0.388850  5.360064  146.169512
,90288  IMrRQ -0.795002  0.764633  6.220779  117.643245
,93570  fSgbT -0.117816  0.511780  6.970150   92.944921
,3478   AsYuH -1.032584  0.200505  8.346149  108.663787
,21407  l9s0m  0.136006  0.852014  3.847119   75.800419
,________________________________________________
,INFORMAÇÕES DA PLANILHA
,<class 'pandas.core.frame.DataFrame'>
,RangeIndex: 100000 entries, 0 to 99999
,Data columns (total 5 columns):
, #   Column   Non-Null Count   Dtype  
,---  ------   --------------   -----  
, 0   id       100000 non-null  object 
, 1   f0       100000 non-n

### 1.2 Região 1

In [4]:
# Dados da Região 1

print('SAMPLE DE DADOS ALEATÓRIOS DA PLANILHA GEO DATA 1')
print(geo_data_1.sample(10))
print('________________________________________________')
print('INFORMAÇÕES DA PLANILHA')
print(geo_data_1.info())
print('________________________________________________')
print('DESCRIÇÃO DE DADOS NUMÉRICOS DA PLANILHA')
print(geo_data_1.describe())
print('________________________________________________')
# Verificando duplicatas
print('LINHAS DUPLICADAS:', geo_data_1.duplicated().sum())

SAMPLE DE DADOS ALEATÓRIOS DA PLANILHA GEO DATA 1
,          id         f0         f1        f2     product
,15288  H0Ekf  -6.982806  -6.185907  5.001323  137.945408
,34109  Ln80k  15.863368  -7.913473  2.999740   80.859783
,61077  uUNWl  -4.047602 -10.491947  4.998842  134.766305
,16422  f3qFI   8.903409   6.531587  5.005035  134.766305
,69900  h2xKh  -2.890962  -6.603359  0.997247   30.132364
,12459  KO23e -13.520853 -12.282182  4.004849  110.992147
,99558  1bLcN  16.536933   2.205227  3.000469   80.859783
,59646  quD86   8.206360 -10.124495  3.993178  107.813044
,91738  jbHOo  14.339139   0.890911  0.000838    0.000000
,40009  ApaRt  -9.006873 -10.057647  1.001407   30.132364
,________________________________________________
,INFORMAÇÕES DA PLANILHA
,<class 'pandas.core.frame.DataFrame'>
,RangeIndex: 100000 entries, 0 to 99999
,Data columns (total 5 columns):
, #   Column   Non-Null Count   Dtype  
,---  ------   --------------   -----  
, 0   id       100000 non-null  object 
, 1  

### 1.3 Região 2

In [5]:
# Dados da Região 2

print('SAMPLE DE DADOS ALEATÓRIOS DA PLANILHA GEO DATA 2')
print(geo_data_2.sample(10))
print('________________________________________________')
print('INFORMAÇÕES DA PLANILHA')
print(geo_data_2.info())
print('________________________________________________')
print('DESCRIÇÃO DE DADOS NUMÉRICOS DA PLANILHA')
print(geo_data_2.describe())
print('________________________________________________')
# Verificando duplicatas
print('LINHAS DUPLICADAS:', geo_data_2.duplicated().sum())

SAMPLE DE DADOS ALEATÓRIOS DA PLANILHA GEO DATA 2
,          id        f0        f1        f2     product
,82713  p54n1 -0.366982 -0.976686  3.951928   54.775952
,96479  8AzR2  1.263642  1.318802  3.822074   76.755152
,8648   v0rRV -0.494490  3.452602  2.812652  155.933035
,49570  kqEh3 -1.475130  3.051962  7.381272  158.845944
,92875  Y5t3j -0.924508  0.683499  3.561003  114.907880
,57283  cQNaV -2.581062 -1.124561  4.264413  160.022111
,43093  GDFEE -1.140589  0.984761  6.254271   36.089276
,51376  17EkA -0.151947 -3.564295  9.900629  147.955649
,8886   mjxSn -0.213662  1.102810  6.168486  123.111372
,92824  6p5JX -0.964059 -5.962889 -1.951914   54.377384
,________________________________________________
,INFORMAÇÕES DA PLANILHA
,<class 'pandas.core.frame.DataFrame'>
,RangeIndex: 100000 entries, 0 to 99999
,Data columns (total 5 columns):
, #   Column   Non-Null Count   Dtype  
,---  ------   --------------   -----  
, 0   id       100000 non-null  object 
, 1   f0       100000 non-n

### 1.4 Conclusões da exploração de dados

A análise exploratória das três regiões confirmou que os conjuntos de dados estão **limpos**, sem valores ausentes e sem linhas duplicadas, dispensando etapas de tratamento ou imputação antes da modelagem.

Ao comparar as estatísticas descritivas, observa-se que:

- As regiões **0 e 2** apresentam volume médio de reservas (`product`) semelhante entre si (**92,50** e **95,00** mil barris, respectivamente), com percentis também próximos;
- A região **1** se diferencia claramente das demais, com volume médio de reservas mais baixo (**68,83** mil barris) e uma proporção maior de poços com reservas muito baixas ou nulas nos percentis inferiores — um primeiro indício de que essa região pode ser menos vantajosa em termos de potencial produtivo.

Quanto à diferença de escala entre as features das regiões — perceptível ao comparar os desvios padrão de `f0`, `f1` e `f2`, bem mais altos na região 1 do que nas regiões 0 e 2 —, cabe uma observação importante: como a **regressão linear ordinária** ajusta seus coeficientes proporcionalmente à escala de cada feature, sem prejuízo à qualidade das predições ou ao RMSE, e como cada região terá seu próprio modelo treinado e avaliado de forma independente (sem comparação direta de coeficientes entre regiões), **a normalização das features não é necessária** neste projeto. Essa prática seria relevante apenas em cenários com um modelo único treinado sobre dados combinados das três regiões, ou com algoritmos sensíveis à escala (como KNN ou modelos com regularização), o que não é o caso aqui.

> Com os dados validados e compreendidos, a próxima etapa é a construção dos modelos preditivos de regressão linear para cada região.

## 2. Modelagem Preditiva

Nesta seção, será construída uma função reutilizável para treinar um modelo de regressão 
linear e avaliar seu desempenho. A função será aplicada às três regiões, evitando a 
duplicação de código.

In [6]:
def treinar_modelo(geo_data):
    """
    Treina um modelo de regressão linear para prever o volume de reservas (product)
    a partir das features f0, f1 e f2.
    
    Parâmetros:
        geo_data (DataFrame): dados da região, contendo colunas id, f0, f1, f2, product.
    
    Retorna:
        result (float): REQM do modelo no conjunto de validação.
        target_valid (Series): valores reais de product no conjunto de validação.
        predictions_valid (array): valores previstos de product no conjunto de validação.
    """
    features_data = geo_data.drop(['id', 'product'], axis=1)
    target_data = geo_data['product']
    
    features_train, features_valid, target_train, target_valid = train_test_split(
        features_data, target_data, test_size=0.25, random_state=12345)
    
    model = LinearRegression()
    model.fit(features_train, target_train)
    predictions_valid = pd.Series(model.predict(features_valid), index=target_valid.index)
    
    result = mean_squared_error(target_valid, predictions_valid) ** 0.5
    
    print('Volume médio previsto de reservas:', predictions_valid.mean())
    print('REQM do modelo:', result)
    
    return result, target_valid, predictions_valid

### 2.1 Aplicação do modelo nas três regiões

In [7]:
print('Região 0:')
result_0, target_valid_0, predictions_valid_0 = treinar_modelo(geo_data_0)

print('\nRegião 1:')
result_1, target_valid_1, predictions_valid_1 = treinar_modelo(geo_data_1)

print('\nRegião 2:')
result_2, target_valid_2, predictions_valid_2 = treinar_modelo(geo_data_2)

Região 0:
,Volume médio previsto de reservas: 92.59256778438035
,REQM do modelo: 37.5794217150813
,
,Região 1:
,Volume médio previsto de reservas: 68.728546895446
,REQM do modelo: 0.893099286775617
,
,Região 2:
,Volume médio previsto de reservas: 94.96504596800489
,REQM do modelo: 40.02970873393434


### 2.2 Investigação: correlação entre features e o alvo

O REQM da região 1 (**0,89**) foi drasticamente menor que o das regiões 0 e 2 (**37,58** e **40,03**, respectivamente). Para entender essa diferença, foi analisada a correlação entre as features (`f0`, `f1`, `f2`) e o alvo (`product`) em cada região.

In [8]:
print('Correlação - Região 0:')
print(geo_data_0.corr())

print('\nCorrelação - Região 1:')
print(geo_data_1.corr())

print('\nCorrelação - Região 2:')
print(geo_data_2.corr())

Correlação - Região 0:
,               f0        f1        f2   product
,f0       1.000000 -0.440723 -0.003153  0.143536
,f1      -0.440723  1.000000  0.001724 -0.192356
,f2      -0.003153  0.001724  1.000000  0.483663
,product  0.143536 -0.192356  0.483663  1.000000
,
,Correlação - Região 1:
,               f0        f1        f2   product
,f0       1.000000  0.182287 -0.001777 -0.030491
,f1       0.182287  1.000000 -0.002595 -0.010155
,f2      -0.001777 -0.002595  1.000000  0.999397
,product -0.030491 -0.010155  0.999397  1.000000
,
,Correlação - Região 2:
,               f0        f1        f2   product
,f0       1.000000  0.000528 -0.000448 -0.001987
,f1       0.000528  1.000000  0.000779 -0.001012
,f2      -0.000448  0.000779  1.000000  0.445871
,product -0.001987 -0.001012  0.445871  1.000000


### 2.3 Comparação com modelo dummy (baseline)

Para contextualizar se o REQM das regiões 0 e 2 — proporcionalmente alto em relação à média de `product` (**≈40%** do valor médio) — representa um modelo pouco útil ou o limite esperado dado os dados disponíveis, foi treinado um **modelo dummy** (baseline), que prevê sempre a média de `product` do conjunto de treino, ignorando as features. Comparar o REQM do modelo de regressão linear com o do dummy permite verificar se o modelo está, de fato, extraindo sinal útil das features.

In [9]:
def treinar_modelo_dummy(geo_data):
    features_data = geo_data.drop(['id', 'product'], axis=1)
    target_data = geo_data['product']
    
    features_train, features_valid, target_train, target_valid = train_test_split(
        features_data, target_data, test_size=0.25, random_state=12345)
    
    dummy_model = DummyRegressor(strategy='mean')
    dummy_model.fit(features_train, target_train)
    dummy_predictions_valid = dummy_model.predict(features_valid)
    
    dummy_result = mean_squared_error(target_valid, dummy_predictions_valid) ** 0.5
    
    print('REQM do modelo dummy:', dummy_result)
    
    return dummy_result
    
print('Região 0 (dummy):')
dummy_result_0 = treinar_modelo_dummy(geo_data_0)

print('\nRegião 1 (dummy):')
dummy_result_1 = treinar_modelo_dummy(geo_data_1)

print('\nRegião 2 (dummy):')
dummy_result_2 = treinar_modelo_dummy(geo_data_2)


Região 0 (dummy):
,REQM do modelo dummy: 44.289591053907365
,
,Região 1 (dummy):
,REQM do modelo dummy: 46.02144533725462
,
,Região 2 (dummy):
,REQM do modelo dummy: 44.90234968510566


### 2.4 Conclusões da modelagem preditiva

Os três modelos de regressão linear apresentaram desempenhos bastante distintos entre si. A região **1** se destacou com um REQM extremamente baixo (**0,89**), enquanto as regiões **0** e **2** apresentaram REQM bem mais elevado (**37,58** e **40,03**, respectivamente) — este último correspondendo a cerca de **40%** do volume médio previsto, o que evidencia uma margem de erro proporcionalmente relevante.

A investigação da matriz de correlação revelou a causa dessa disparidade: na região 1, a feature `f2` apresenta correlação **quase perfeita** com `product` (**0,999**), enquanto nas regiões 0 e 2 a correlação mais forte com o alvo (também de `f2`) é apenas **moderada** (**0,44** e **0,45**, respectivamente). Isso indica que a região 1 possui uma relação quase determinística entre uma de suas características geológicas e o volume de reservas, o que explica a altíssima precisão do modelo nessa região.

Para avaliar se o desempenho dos modelos nas regiões 0 e 2 — aparentemente fraco em termos absolutos — reflete um modelo pouco útil ou o limite de informação disponível nos dados, foi realizada uma comparação com um modelo dummy. Em todas as três regiões, o modelo de regressão linear **superou o dummy**:

| Região | REQM modelo | REQM dummy |
|---|---|---|
| 0 | 37,58 | 44,29 |
| 1 | 0,89 | 46,02 |
| 2 | 40,03 | 44,90 |

Isso confirma que, mesmo nas regiões com maior incerteza residual, o modelo está de fato extraindo sinal útil das features — e não apenas replicando a média geral.

> Conclui-se que os três modelos são válidos e superiores a um baseline ingênuo, mas com níveis de confiabilidade distintos: a região 1 oferece previsões muito mais precisas que as regiões 0 e 2. Essa diferença de incerteza entre regiões é um fator relevante a ser considerado nas próximas etapas, especialmente na análise de risco via bootstrapping, já que maior precisão nas predições tende a se refletir em menor variabilidade nos cenários simulados de lucro.

## 3. Preparação para o Cálculo de Lucro

Antes de simular a seleção de poços e calcular o lucro potencial, é necessário estabelecer o volume mínimo de reservas (**breakeven**) que cada poço precisa produzir para que o investimento não resulte em prejuízo, dado o orçamento e a receita por unidade de `product`.

In [10]:
# Variáveis do negócio
budget = 100_000_000        # orçamento total disponível, em dólares
wells_count = 200           # número de poços a serem desenvolvidos
revenue_per_unit = 4500     # receita gerada por unidade de product (mil barris), em dólares

# Cálculo do orçamento disponível por poço e do breakeven (em unidades de product)
budget_per_well = budget / wells_count
breakeven = budget_per_well / revenue_per_unit

print('Orçamento por poço:', budget_per_well)
print('Breakeven (unidades de product necessárias por poço):', breakeven)

Orçamento por poço: 500000.0
,Breakeven (unidades de product necessárias por poço): 111.11111111111111


### 3.1 Comparação do breakeven com o volume médio por região

In [11]:
mean_product_0 = predictions_valid_0.mean()
mean_product_1 = predictions_valid_1.mean()
mean_product_2 = predictions_valid_2.mean()

print('Volume médio previsto - Região 0:', mean_product_0)
print('Volume médio previsto - Região 1:', mean_product_1)
print('Volume médio previsto - Região 2:', mean_product_2)
print('Breakeven necessário por poço:', breakeven)

Volume médio previsto - Região 0: 92.59256778438035
,Volume médio previsto - Região 1: 68.728546895446
,Volume médio previsto - Região 2: 94.96504596800489
,Breakeven necessário por poço: 111.11111111111111


### 3.2 Conclusões da preparação para o cálculo de lucro

O **breakeven** calculado — o volume mínimo de reservas que cada poço precisa produzir para não gerar prejuízo — é de aproximadamente **111,1 mil barris**. Comparando esse valor com o volume médio previsto de `product` em cada região:

| Região | Volume médio previsto | Breakeven |
|---|---|---|
| 0 | 92,59 | 111,1 |
| 1 | 68,73 | 111,1 |
| 2 | 94,97 | 111,1 |

**Todas as três regiões apresentam volume médio abaixo do breakeven.**

Isso significa que, se a empresa selecionasse 200 poços de forma **aleatória** em qualquer uma das regiões, o resultado esperado seria **prejuízo**, já que a média geral de reservas não é suficiente para cobrir o custo por poço. Esse resultado evidencia a necessidade do modelo preditivo: a estratégia de negócio só se torna viável ao selecionar, dentro de cada região, os poços com **maior volume previsto** — e não uma amostra aleatória —, concentrando o desenvolvimento nos pontos mais promissores identificados pelo modelo.

> A próxima etapa consiste em simular essa seleção: escolher os 200 poços com maior volume previsto em cada região (entre uma amostra de 500) e calcular o lucro potencial resultante.

## 4. Cálculo de Lucro

Nesta seção, será construída uma função para calcular o lucro potencial de uma seleção de poços, com base nas predições do modelo. A lógica simula a decisão real da empresa: os poços são escolhidos com base no valor **previsto** pelo modelo (a única informação disponível antes de perfurar), mas o lucro é calculado com base no valor **real** de `product` desses mesmos poços — já que é isso que efetivamente seria extraído na prática.

In [12]:
def calcular_lucro(target, predictions):
    """
    Calcula o lucro potencial ao selecionar os 200 poços com maior valor previsto
    de reservas, entre um conjunto de poços avaliados.
    
    Parâmetros:
        target (Series): valores reais de product dos poços avaliados.
        predictions (Series): valores previstos de product para os mesmos poços,
                               com o mesmo índice de target.
    
    Retorna:
        lucro (float): lucro obtido com os 200 poços selecionados, em dólares.
    """
    # 1. Ordena os poços pela predição (do maior para o menor) e pega os índices dos 200 melhores
    indices_ordenados = predictions.sort_values(ascending=False).index
    top_200_indices = indices_ordenados[:200]
    
    # 2. Recupera os valores REAIS de product apenas para esses 200 poços selecionados
    volumes_reais_top_200 = target.loc[top_200_indices]
    
    # 3. Soma o volume total de reservas que será extraído dos 200 poços escolhidos
    volume_total = volumes_reais_top_200.sum()
    
    # 4. Calcula a receita gerada por esse volume
    receita = volume_total * revenue_per_unit
    
    # 5. Calcula o lucro: receita obtida menos o investimento total
    lucro = receita - budget
    
    return lucro

### 4.1 Aplicação da função nas três regiões

A função é aplicada utilizando o conjunto de validação completo de cada região (cerca de **25 mil poços**), selecionando os 200 melhores segundo a predição do modelo.

In [13]:
lucro_0 = calcular_lucro(target_valid_0, predictions_valid_0)
print(f'Lucro potencial - Região 0: {lucro_0:,.2f}')

lucro_1 = calcular_lucro(target_valid_1, predictions_valid_1)
print(f'Lucro potencial - Região 1: {lucro_1:,.2f}')

lucro_2 = calcular_lucro(target_valid_2, predictions_valid_2)
print(f'Lucro potencial - Região 2: {lucro_2:,.2f}')

Lucro potencial - Região 0: 33,208,260.43
,Lucro potencial - Região 1: 24,150,866.97
,Lucro potencial - Região 2: 27,103,499.64


### 4.2 Conclusões do cálculo de lucro

Ao selecionar os 200 poços com maior valor previsto entre a totalidade do conjunto de validação de cada região, o lucro potencial calculado foi:

| Região | Lucro potencial (US$) |
| --- | --- |
| 0 | 33.208.260,43 |
| 1 | 24.150.866,97 |
| 2 | 27.103.499,64 |

Considerando apenas esse cenário isolado, a região **0** apresenta o maior lucro potencial, seguida pela região 2 e, por último, a região 1 — resultado consistente com o fato de a região 1 possuir o menor volume médio de reservas entre as três.

Chama atenção, no entanto, que a região 2 apresenta volume médio de reservas **ligeiramente maior** que a região 0 (94,97 contra 92,59 mil barris), mas resultou em lucro potencial **menor**. Como as duas regiões possuem desvio padrão de `product` e força de correlação das features com o alvo semelhantes, essa diferença não parece ter uma causa estrutural clara, podendo refletir a **variabilidade natural** da amostra específica de poços que compôs o conjunto de validação de cada região.

Uma hipótese alternativa, a de que a região 0 possuiria outliers de valor elevado concentrados entre os poços de maior predição, também foi avaliada — porém os valores de máximo e do percentil 75% de `product` são, na verdade, **maiores na região 2**, o que contraria essa hipótese.

Também vale destacar que a região 1, apesar de possuir o modelo preditivo mais preciso (menor REQM, devido à forte correlação entre a feature f2 e o volume de reservas), apresentou o **menor** lucro potencial entre as três — evidenciando que **precisão do modelo** e **potencial de lucro** são fatores distintos.

> Como o resultado desta seção se baseia em uma única seleção, feita sobre um grande conjunto de 25 mil poços — cenário mais favorável do que a realidade operacional descrita no projeto, na qual cada estudo avalia apenas 500 poços por vez —, a próxima seção avalia como o lucro e o risco de prejuízo se comportam sob essa condição mais realista, e se a vantagem observada da região 0 se mantém de forma consistente ao longo de 1.000 simulações de bootstrapping.

## 5. Avaliação de Risco via Bootstrapping

Como o resultado da Seção 4 se baseou em uma **única seleção**, feita sobre o conjunto de validação inteiro (25 mil poços) — um cenário mais favorável do que a operação real da empresa —, esta seção simula a condição descrita nas regras de negócio: em cada tentativa de exploração, apenas **500 poços** são estudados por vez, dos quais os **200 melhores** são selecionados.

Para entender a distribuição de resultados possíveis nesse cenário mais realista, será aplicada a técnica de **bootstrapping**: o processo de sortear 500 poços (com reposição) e calcular o lucro dos 200 melhores será repetido **1.000 vezes**, gerando uma distribuição de lucros para cada região. A partir dela, será possível calcular o **lucro médio esperado**, o **intervalo de confiança de 95%** e o **risco de prejuízo** (probabilidade de lucro negativo).

In [14]:
# Gerador de números aleatórios, criado uma única vez para garantir reprodutibilidade
# e independência entre os sorteios das diferentes regiões
state = np.random.RandomState(12345)

def bootstrap_lucro(predictions, target):
    """
    Simula 1.000 cenários de exploração, sorteando 500 poços (com reposição) por vez
    e calculando o lucro dos 200 melhores poços selecionados em cada sorteio.
    
    Parâmetros:
        predictions (Series): valores previstos de product no conjunto de validação.
        target (Series): valores reais de product no conjunto de validação,
                          com o mesmo índice de predictions.
    
    Retorna:
        Series com os 1.000 valores de lucro simulados.
    """

    lucros = []
    for i in range(1000):
        predictions_subsample = predictions.sample(n=500, replace=True, random_state=state)
        target_subsample = target.loc[predictions_subsample.index]
        predictions_subsample = predictions_subsample.reset_index(drop=True)
        target_subsample = target_subsample.reset_index(drop=True)
        lucro = calcular_lucro(target_subsample, predictions_subsample)
        lucros.append(lucro)
    return pd.Series(lucros)

### 5.1 Aplicação do bootstrapping nas três regiões

In [15]:
lucros_0 = bootstrap_lucro(predictions_valid_0, target_valid_0)
lucros_1 = bootstrap_lucro(predictions_valid_1, target_valid_1)
lucros_2 = bootstrap_lucro(predictions_valid_2, target_valid_2)

### 5.2 Cálculo de lucro médio, intervalo de confiança e risco de prejuízo

Para cada região, calcula-se: o **lucro médio** das 1.000 simulações, o **intervalo de confiança de 95%** (delimitado pelos percentis 2,5% e 97,5% da distribuição de lucros) e o **risco de prejuízo** (proporção de simulações com lucro negativo, expressa em porcentagem).

In [16]:
def resumir_risco(lucros):
    """
    Calcula as métricas de risco e retorno a partir de uma distribuição de lucros
    simulados via bootstrapping.
    
    Parâmetros:
        lucros (Series): valores de lucro simulados (uma linha por simulação de bootstrap).
    
    Retorna:
        media (float): lucro médio da distribuição.
        ic_inferior (float): limite inferior do intervalo de confiança de 95%.
        ic_superior (float): limite superior do intervalo de confiança de 95%.
        risco_prejuizo (float): percentual de simulações com lucro negativo.
    """
    media = lucros.mean()
    ic_inferior = lucros.quantile(0.025)
    ic_superior = lucros.quantile(0.975)
    
    # Cálculo do risco de prejuízo, dividido em passos:
    simulacoes_com_prejuizo = lucros < 0                            # True onde o lucro foi negativo
    total_simulacoes_com_prejuizo = simulacoes_com_prejuizo.sum()   # conta quantos True existem
    total_simulacoes = len(lucros)                                  # total de simulações realizadas
    risco_prejuizo = total_simulacoes_com_prejuizo / total_simulacoes * 100  # converte para %
    
    print(f'Lucro médio: {media:,.2f}')
    print(f'Intervalo de confiança 95%: [{ic_inferior:,.2f}, {ic_superior:,.2f}]')
    print(f'Risco de prejuízo: {risco_prejuizo:.2f}%')
    
    return media, ic_inferior, ic_superior, risco_prejuizo

In [17]:
print('Região 0:')
media_0, ic_inf_0, ic_sup_0, risco_0 = resumir_risco(lucros_0)

print('\nRegião 1:')
media_1, ic_inf_1, ic_sup_1, risco_1 = resumir_risco(lucros_1)

print('\nRegião 2:')
media_2, ic_inf_2, ic_sup_2, risco_2 = resumir_risco(lucros_2)

Região 0:
,Lucro médio: 3,961,649.85
,Intervalo de confiança 95%: [-1,112,155.46, 9,097,669.42]
,Risco de prejuízo: 6.90%
,
,Região 1:
,Lucro médio: 4,611,558.17
,Intervalo de confiança 95%: [780,508.11, 8,629,520.60]
,Risco de prejuízo: 0.70%
,
,Região 2:
,Lucro médio: 3,929,504.75
,Intervalo de confiança 95%: [-1,122,276.25, 9,345,629.15]
,Risco de prejuízo: 6.50%


### 5.3 Conclusões da avaliação de risco

A simulação de bootstrapping, com **1.000 repetições** do processo real de exploração (sorteio de 500 poços e seleção dos 200 melhores), produziu os seguintes resultados, em dólares:

| Região | Lucro médio | Limite inferior (IC 95%) | Limite superior (IC 95%) | Risco de prejuízo |
| --- | --- | --- | --- | --- |
| 0 | 3.961.649,85 | -1.112.155,46 | 9.097.669,42 | 6,90% |
| 1 | **4.611.558,17** | 780.508,11 | 8.629.520,60 | **0,70%** |
| 2 | 3.929.504,75 | -1.122.276,25 | 9.345.629,15 | 6,50% |

Aplicando o critério de negócio da OilyGiant — considerar apenas regiões com risco de prejuízo **inferior** a 2,5% —, **apenas a região 1 atende ao critério**. As regiões 0 e 2 são eliminadas, com risco de prejuízo de 6,90% e 6,50%, respectivamente — bem acima do limite estabelecido.

Além do risco muito mais baixo, a região 1 também apresenta o maior lucro médio entre as três (**4.611.558,17**) e é a única cujo intervalo de confiança de 95% permanece **inteiramente positivo** (780.508,11 a 8.629.520,60). Já as regiões 0 e 2 têm limite inferior **negativo**, o que significa que, em uma parcela relevante das simulações, o resultado seria de fato prejuízo — evidenciando visualmente por que seu risco de prejuízo é tão mais alto.

Essa maior segurança da região 1 está diretamente relacionada ao que foi observado na Seção 2: seu modelo apresentou REQM extremamente baixo, devido à forte correlação entre a feature `f2` e o volume de reservas. Essa precisão elevada do modelo se traduz, na prática, em decisões de perfuração mais confiáveis e menos sujeitas à variação aleatória da amostra sorteada em cada simulação.

> **Essa recomendação diverge da conclusão obtida na Seção 4**, quando a região 0 apresentava o maior lucro potencial ao selecionar os 200 melhores poços entre os 25 mil do conjunto de validação completo. Essa mudança de resultado ocorre porque as duas análises respondem a perguntas diferentes: a Seção 4 estima o lucro em um cenário hipotético e mais favorável, no qual a empresa teria acesso a 25 mil poços de uma só vez para escolher os 200 melhores; já a Seção 5 simula a operação real descrita nas regras de negócio, na qual apenas 500 poços são avaliados por vez. Ao repetir esse processo mais realista 1.000 vezes, o bootstrapping revela que o desempenho da região 0 não se mantém consistente quando o tamanho do lote de exploração é reduzido para 500 — a vantagem observada na Seção 4 estava associada a ter um conjunto muito maior de poços para escolher os melhores, algo que não reflete a forma como a empresa realmente opera.

## 6. Recomendação Final

Com base na análise completa — modelagem preditiva, cálculo de lucro potencial e avaliação de risco via bootstrapping —, **recomenda-se a Região 1** para o desenvolvimento dos novos poços de petróleo pela OilyGiant.

A região 1 é a **única** das três que atende ao critério de risco estabelecido pela empresa (risco de prejuízo inferior a 2,5%), combinando:

- **Risco de prejuízo de apenas 0,70%**, muito abaixo do limite de 2,5% e substancialmente menor que o das regiões 0 (6,90%) e 2 (6,50%);
- **Maior lucro médio esperado** entre as três regiões (4.611.558,17 dólares);
- **Intervalo de confiança de 95% inteiramente positivo**, ao contrário das regiões 0 e 2, cujos intervalos incluem cenários de prejuízo;
- A **maior confiabilidade preditiva** do modelo, dado o REQM observado na Seção 2.

Vale destacar que a região 1 possui o **menor volume médio de reservas** entre as três regiões (68,73 mil barris, contra 92,59 e 94,97 nas regiões 0 e 2). Ainda assim, é a escolha recomendada — o que reforça que a decisão não se baseia no maior potencial bruto de petróleo, e sim na combinação entre retorno esperado e previsibilidade do resultado, que é justamente o critério que a OilyGiant definiu como prioritário.

Isso torna a Região 1 a escolha claramente mais sólida do ponto de vista financeiro e estatístico para o desenvolvimento dos novos poços de petróleo.